<a href="https://colab.research.google.com/github/madhugadi/LLM-fine-tuning/blob/main/knowledge_Distillation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install --upgrade datasets fsspec transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 7.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 39.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.9/53.9 MB 73.6 MB/s  0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 23.0.1
    Uninstalling pyarrow-23.0.1:
      Successfully uninstalled pyarrow-23.0.1
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.8.5
    Uninstalling datasets-4.8.5:
      Successfully uninstalled datasets-4.8.5
  Attempting uninstall: transformers
    Found existing installation: transformers 5.20.2
    Uninstalling transformers-5.20.2:
      Successfully uninstalled transformers-5.20.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [transformers]
ERROR: pip's dependency resolver does not curr

In [62]:
# Suppress warning messages (such as future deprecation notices) for cleaner output
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

In [63]:
# Import general deep learning frameworks and utilities
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# Import Hugging Face datasets and transformers tools
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

In [64]:
# ----- General Hyperparameters -----
batch_size   = 16    # Size of training batches
lr           = 5e-5    # Learning rate for the optimizer
epochs       = 1     # Total training runs over the dataset
temperature  = 2.0   # Softens probabilities for knowledge distillation
alpha_soft   = 0.5   # Weighting factor for distillation loss versus standard loss
max_len      = 128   # Maximum sequence length for tokenization

# Configure PyTorch to utilize a GPU if one is available
device       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [65]:
# Load the sentiment subset of the cardiffnlp/tweet_eval dataset
raw = load_dataset("cardiffnlp/tweet_eval", "sentiment")

In [8]:
label_feature = raw["train"].features["label"]

In [9]:
print("Label names:", label_feature.names)

Label names: ['negative', 'neutral', 'positive']


In [66]:
# Shuffle the training split and select a 2,500 sample subset for faster training
train = raw['train'].shuffle(seed=42).select(range(2500))

In [11]:
val   = raw['validation']

In [67]:
# Initialize the pre-trained BERT-base tokenizer for encoder distillation
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

In [68]:
# Define the tokenization helper function
def tokenize(example):
    # Tokenize input text with truncation to the defined max length
    return tokenizer(example["text"], truncation=True, max_length=max_len)

In [14]:
# Tokenize & remove original text column
tokenized = {}

In [69]:
# Tokenize training and validation splits and drop the raw 'text' column
tokenized = {}
tokenized['train'] = train.map(tokenize, batched=True, remove_columns=['text'])
tokenized['validation'] = val.map(tokenize, batched=True, remove_columns=['text'])

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [70]:
# Dynamically pad inputs to the maximum sequence length in each batch
collator = DataCollatorWithPadding(tokenizer, pad_to_multiple_of=8)

In [71]:
# Build the training DataLoader with batching and shuffling enabled
train_dl = DataLoader(tokenized['train'], batch_size=batch_size, shuffle=True, collate_fn=collator)

In [72]:
# Build the validation DataLoader
val_dl = DataLoader(tokenized['validation'], batch_size=batch_size, shuffle=False, collate_fn=collator)

In [19]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

In [20]:
num_labels = 3

In [73]:
# Load the large-scale Teacher model (BERT-Large) for sequence classification
teacher = AutoModelForSequenceClassification.from_pretrained(
    "bert-large-uncased", num_labels=num_labels).to(device)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-large-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [74]:
# Load the smaller Student model (BERT-Base) to receive the distillation
student = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_labels).to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [75]:
# Freeze teacher parameters to save memory and prevent weights update
for p in teacher.parameters():
    p.requires_grad = False
teacher.eval()

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 1024, padding_idx=0)
      (position_embeddings): Embedding(512, 1024)
      (token_type_embeddings): Embedding(2, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-23): 24 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
              (LayerNorm): LayerNorm((1024,

In [24]:
ce_loss = nn.CrossEntropyLoss()

In [25]:
kl_loss = nn.KLDivLoss(reduction="batchmean")

In [26]:
optimizer = optim.AdamW(student.parameters(), lr=lr)

In [27]:
from transformers import get_scheduler

In [32]:
lr_scheduler = get_scheduler(
    name="linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=len(train_dl) * epochs,
)

In [33]:
from tqdm.auto import tqdm

In [76]:
# Core training function for a single epoch using Knowledge Distillation
def distill_epoch():
    student.train()
    pbar = tqdm(train_dl, desc="Train")
    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        attention = batch["attention_mask"].to(device)
        labels    = batch["labels"].to(device)

        # Get soft probabilities from the static Teacher
        with torch.no_grad():
            t_logits = teacher(input_ids, attention_mask=attention).logits
            t_soft   = torch.softmax(t_logits / temperature, dim=1)

        # Get Student log probabilities
        s_logits = student(input_ids, attention_mask=attention).logits
        s_soft   = torch.log_softmax(s_logits / temperature, dim=1)

        # Calculate Soft Distillation loss (KL Divergence) and Hard Target loss (CrossEntropy)
        loss_soft = kl_loss(s_soft, t_soft) * (temperature ** 2)
        loss_hard = ce_loss(s_logits, labels)
        loss      = alpha_soft * loss_soft + (1 - alpha_soft) * loss_hard

        # Backpropagation and optimizer step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        pbar.set_postfix({"loss": f"{loss.item():.4f}"})

In [77]:
# Evaluation function to measure accuracy on the validation split
def evaluate():
    student.eval()
    correct = total = 0
    with torch.no_grad():
        for batch in val_dl:
            ids  = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            lbl  = batch["labels"].to(device)
            out  = student(ids, attention_mask=attn).logits
            pred = out.argmax(dim=1)
            correct += (pred == lbl).sum().item()
            total   += lbl.size(0)
    return round(correct / total * 100, 2)

In [36]:
for ep in range(1, epochs + 1):
    distill_epoch()
    acc = evaluate()
    print(f"Epoch {ep}/{epochs} | Validation Accuracy: {acc}%")

Train:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 1/1 | Validation Accuracy: 68.65%


In [78]:
# Save fine-tuned student model weights and tokenizer to a local directory
student.save_pretrained("distilled_student_model")
tokenizer.save_pretrained("distilled_student_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('distilled_student_model/tokenizer_config.json',
 'distilled_student_model/tokenizer.json')

In [79]:
# Standard interface function for predicting outcomes and logging latency
def predict_and_evaluate(model, name, test_dl):
    model.eval()
    all_preds, all_labels = [], []
    start_time = time.time()

    with torch.no_grad():
        for batch in test_dl:
            ids = batch["input_ids"].to(device)
            attn = batch["attention_mask"].to(device)
            lbls = batch["labels"].to(device)

            logits = model(ids, attention_mask=attn).logits
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(lbls.cpu().tolist())

    total_time = time.time() - start_time
    acc = accuracy_score(all_labels, all_preds)
    avg_time = total_time / len(test_dl.dataset)

    print(f"\n {name}")
    print(f" Accuracy: {acc*100:.2f}%")
    print(f" Total Inference Time: {total_time:.2f} sec")
    print(f" Avg Time per Sample: {avg_time:.4f} sec")
    return acc, total_time, avg_time

In [40]:
# --------- Load test set ---------
test = load_dataset("cardiffnlp/tweet_eval", "sentiment", split="test[:500]")  # sample test
tokenized_test = test.map(tokenize, batched=True, remove_columns=["text"])
tokenized_test.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
test_dl = DataLoader(tokenized_test, batch_size=batch_size, shuffle=False, collate_fn=collator)

Map:   0%|          | 0/500 [00:00<?, ? examples/s]

In [41]:
# --------- Compare Teacher vs Student ---------
predict_and_evaluate(teacher, name="TEACHER (BERT-Large)", test_dl=test_dl)
predict_and_evaluate(student, name="STUDENT (Distilled BERT)", test_dl=test_dl)



 TEACHER (BERT-Large)
 Accuracy: 30.00%
 Total Inference Time: 4.60 sec
 Avg Time per Sample: 0.0092 sec

 STUDENT (Distilled BERT)
 Accuracy: 65.80%
 Total Inference Time: 1.49 sec
 Avg Time per Sample: 0.0030 sec


(0.658, 1.4862654209136963, 0.0029725308418273927)

In [42]:
## LLM fine tuning

In [43]:
!pip install transformers accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.1 MB/s  0:00:01


In [44]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn as nn
import torch.optim as optim

In [45]:
teacher_id = "microsoft/phi-2"
student_id = "microsoft/phi-1_5"

In [46]:
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

config.json:   0%|          | 0.00/735 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.34k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

...

vocab.json:   0%|          | 0.00/798k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/1.08k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.11M [00:00<?, ?B/s]

In [47]:
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

In [87]:
# Configure quantization parameters
from transformers import BitsAndBytesConfig
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# Load the causal LLM Teacher (Phi-2) using 8-bit quantization with a distinct variable name
phi_teacher = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    device_map="auto",
    quantization_config=quantization_config,
)

Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

In [50]:
teacher_id = "microsoft/phi-2"
student_id = "microsoft/phi-1_5"

In [51]:
teacher_tokenizer = AutoTokenizer.from_pretrained(teacher_id)
student_tokenizer = AutoTokenizer.from_pretrained(student_id)

In [52]:
if teacher_tokenizer.pad_token is None:
    teacher_tokenizer.pad_token = teacher_tokenizer.eos_token
if student_tokenizer.pad_token is None:
    student_tokenizer.pad_token = student_tokenizer.eos_token

In [61]:
teacher = AutoModelForCausalLM.from_pretrained(
    teacher_id,
    device_map="auto",
    load_in_8bit=True,
)

TypeError: PhiForCausalLM.__init__() got an unexpected keyword argument 'load_in_8bit'

In [54]:
for p in teacher.parameters():
    print(p)

Parameter containing:
tensor([[ 1.6998e-02, -1.3275e-02,  2.0309e-02,  ...,  1.7822e-02,
          5.1346e-03, -7.7343e-04],
        [ 9.7351e-03,  5.1636e-02,  1.5656e-02,  ..., -6.7329e-03,
          6.9389e-03, -1.1322e-02],
        [-4.6387e-02, -9.0942e-03, -1.1349e-03,  ..., -3.0945e-02,
          3.8940e-02,  1.3847e-02],
        ...,
        [-5.9605e-08,  5.9605e-08, -5.9605e-08,  ...,  1.5140e-05,
         -1.1206e-05,  1.7762e-05],
        [-0.0000e+00, -5.9605e-08, -1.1921e-07,  ..., -2.5094e-05,
          3.7730e-05,  2.0683e-05],
        [ 0.0000e+00, -5.9605e-08,  5.9605e-08,  ..., -1.6034e-05,
         -1.5676e-05, -3.5882e-05]], device='cuda:0', dtype=torch.float16,
       requires_grad=True)
Parameter containing:
Parameter(Int8Params([[ 26,  -5, -23,  ..., -30,  15,  11],
            [-37, -16, -10,  ...,  15,   0, -26],
            [-22, -11,  -4,  ...,   0,   9,   9],
            ...,
            [-35, -26,   5,  ...,  20,  12,  34],
            [-37,  50,  -2,  ...

In [88]:
# Freeze quantized causal Teacher parameters safely using distinct variable name
phi_teacher.eval()
for p in phi_teacher.parameters():
    p.requires_grad = False

In [97]:
import gc
import torch

# Safely clear the old student model from memory first
try:
    del phi_student
except NameError:
    pass
gc.collect()
torch.cuda.empty_cache()

# Load the student Causal LM in half precision (float16) to save VRAM
phi_student = AutoModelForCausalLM.from_pretrained(
    student_id,
    device_map="auto",
    torch_dtype=torch.float16
)

# Enable gradient checkpointing on the student to drastically lower activation memory
phi_student.gradient_checkpointing_enable()

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/341 [00:00<?, ?it/s]

In [95]:
# ----- Distillation Hyperparams for Causal LLM -----
temperature = 2.0
alpha_soft  = 0.7
ce_loss     = nn.CrossEntropyLoss(ignore_index=student_tokenizer.pad_token_id)
kl_loss     = nn.KLDivLoss(reduction="batchmean")

# Use 8-bit Adam optimizer from bitsandbytes to reduce optimizer state footprint by 75%
import bitsandbytes as bnb
optimizer   = bnb.optim.Adam8bit(phi_student.parameters(), lr=2e-5)

In [96]:
# Explicitly release old sequence classification model references to free GPU VRAM
import gc
import torch

teacher = None
student = None
gc.collect()
torch.cuda.empty_cache()

# Distillation training loop iterating over text prompts with Mixed Precision (AMP) to save memory
for prompt in prompts:
    # Clear cache before each prompt forward pass
    torch.cuda.empty_cache()

    # Tokenize the prompt text using teacher and student structures
    t_inputs = teacher_tokenizer(prompt, return_tensors="pt", padding=True).to(phi_teacher.device)
    s_inputs = student_tokenizer(prompt, return_tensors="pt", padding=True).to(phi_student.device)

    # Get soft targets from the quantized causal Teacher
    with torch.no_grad():
        with torch.cuda.amp.autocast():
            t_logits = phi_teacher(**t_inputs).logits
            if t_logits.ndim == 2:
                t_logits = t_logits.unsqueeze(1)
            # Shift logits for causal LM training (predict next token)
            t_logits = t_logits[:, :-1, :]
            t_soft = torch.softmax(t_logits / temperature, dim=-1)
            t_soft = torch.clamp(t_soft, min=1e-8)

    # Get predictions from Student Causal LM inside AMP context
    with torch.cuda.amp.autocast():
        s_logits = phi_student(**s_inputs).logits
        if s_logits.ndim == 2:
            s_logits = s_logits.unsqueeze(1)
        s_logits = s_logits[:, :-1, :]
        s_log_soft = torch.log_softmax(s_logits / temperature, dim=-1)

        # Align target labels to the student sequence length
        labels = s_inputs["input_ids"]
        if labels.ndim == 1:
            labels = labels.unsqueeze(0)
        if labels.size(1) > 1:
            labels = labels[:, 1:].contiguous()

        # Ensure s_logits sequence dimension matches labels sequence dimension
        min_seq_len = min(s_logits.size(1), labels.size(1), t_soft.size(1))
        s_logits = s_logits[:, :min_seq_len, :].contiguous()
        labels = labels[:, :min_seq_len].contiguous()
        t_soft = t_soft[:, :min_seq_len, :].contiguous()
        s_log_soft = s_log_soft[:, :min_seq_len, :].contiguous()

        # Compute individual loss functions with aligned, flattened inputs
        loss_hard = ce_loss(s_logits.view(-1, s_logits.size(-1)), labels.view(-1))
        loss_soft = kl_loss(s_log_soft, t_soft) * (temperature ** 2)

        # Aggregate weighted objectives
        loss = alpha_soft * loss_soft + (1 - alpha_soft) * loss_hard

    if torch.isnan(loss):
        print("☑þ NaN detected on prompt:", prompt[:50])
        continue

    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    print(f"Prompt: {prompt[:40]}..., Loss: {loss.item():.4f}")

OutOfMemoryError: CUDA out of memory. Tried to allocate 200.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 15.81 MiB is free. Including non-PyTorch memory, this process has 14.54 GiB memory in use. Of the allocated memory 14.03 GiB is allocated by PyTorch, and 375.03 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [92]:
# Save the correct causal student model weights
phi_student.save_pretrained("distilled_phi1_5")
student_tokenizer.save_pretrained("distilled_phi1_5")
print("Distillation complete. Model saved in 'distilled_phi1_5/'")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Distillation complete. Model saved in 'distilled_phi1_5/'


In [60]:
# Define sample prompts to run the distillation
prompts = [
    "Deep learning is a subset of machine learning that",
    "The capital of France is Paris, which is known for",
    "To train a neural network effectively, we need to",
    "Artificial intelligence will shape the future by",
    "Data science involves extracting insights from"
]